# Author name match and merge

Disclaimer: this file is currently using data scraped around Oct-Dec 2025, and might not reflect the most recently updated author lists off either SciVal or OpenAlex. This will be updated with newer data ASAP. 

In [2]:
# import libraries 

import pandas as pd
import numpy as np
import json
import ast
import fuzzywuzzy
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from datetime import datetime
from tqdm import tqdm 
from collections import defaultdict


c:\Users\tania\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
# script to aggregate dicts for later 

def merge_year_dict_lists(list1, list2):
    combined = list1 + list2
    merged = defaultdict(lambda: {
        "year": None,
        "works_count": 0,
        "oa_works_count": 0,
        "cited_by_count": 0
    })
    
    for item in combined:
        year = item["year"]
        merged[year]["year"] = year
        merged[year]["works_count"] += item.get("works_count", 0)
        merged[year]["oa_works_count"] += item.get("oa_works_count", 0)
        merged[year]["cited_by_count"] += item.get("cited_by_count", 0)
    
    # return sorted list 
    return sorted(merged.values(), key=lambda x: x["year"])

### 1. Read and clean OpenAlex author data for SFU

For the sake of simplicity, limit only to authors who have at least 1 publication since 2020. 

In [4]:
# read OpenAlex data 
oa_authors_raw = pd.read_csv("sfu_all_authors_2.csv")

#oa_authors_raw.head()

In [5]:
# clean OpenAlex data 

oa_authors_clean = oa_authors_raw

# check last year published and number of publications
oa_authors_clean['counts_clean'] = oa_authors_clean['counts_by_year'].apply(lambda x: ast.literal_eval(x))
oa_authors_clean['latest_year'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['year']) # -1 index grabs last year
oa_authors_clean['pubs_in_latest_yr'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['works_count'])

# get rid of unnecessary cols
oa_authors_clean = oa_authors_clean.drop(columns=['topic_share', 'x_concepts', 'counts_by_year', 'updated_date', 'created_date'])

# save only authors who have published since 2020
oa_authors_old = oa_authors_clean[oa_authors_clean['latest_year'] < 2020].reset_index().drop(columns=['index']) # keep for future reference if necessary 
oa_authors_clean = oa_authors_clean[oa_authors_clean['latest_year'] >= 2020].reset_index().drop(columns=['index']) # update clean table

# clean id col 
oa_authors_clean['id'] = oa_authors_clean['id'].apply(lambda x: x.lstrip('https://openalex.org/'))
oa_authors_clean['orcid'] = oa_authors_clean['orcid'].apply(lambda x: str(x).lstrip('https://orcid.org/'))

# deal with names
oa_authors_clean['name_raw'] = oa_authors_clean['display_name'].apply(lambda x: str(x).replace('.', '').replace("'", "").strip(' ').lower())
oa_authors_clean['last_name'] = oa_authors_clean['name_raw'].apply(lambda x: str(x).split(' ')[-1].strip(' '))
oa_authors_clean['last_init'] = oa_authors_clean['last_name'].apply(lambda x: str(x)[0])
oa_authors_clean['first_names'] = oa_authors_clean['name_raw'].apply(lambda x: str(str(x).split(' ')[0:-1]))
oa_authors_clean['potential_matches'] = [[] for _ in range(len(oa_authors_clean))]

# testing 
print(oa_authors_clean['counts_clean'][1][-1]['year'])
print(oa_authors_clean['latest_year'].unique())
print(oa_authors_clean['pubs_in_latest_yr'].unique())

#oa_authors_clean

2026
[2026 2025 2022 2024 2023 2020 2021 2027 2029]
[  44    1   64    2    5    9   18   27    8   10 2057 2021   12    6
    3    7   30   77    4  148   87   74   97   98   41  101   85  108
   40   83  100  104   17  112   76   91   94   14   99   29   25   31
   28   72   24   20   54   43   79   53   15   35   33   23   19   37
   82   48   26   57   32   69   46   11   61   42   34   60   78   45
   59   52   50  102   16   62   55   49   93   13   22  188   39   21
   58   96  132   73   56  144   70   75   66   67   71   68   36   38
  242  105  169   47   63]


In [6]:
len(oa_authors_clean)

20049

In [40]:
oa_authors_raw[oa_authors_raw['display_name'].str.contains("Pasquier")]

,id,orcid,display_name,raw_author_names,full_name,works_count,cited_by_count,summary_stats,ids,affiliations,...,x_concepts,counts_by_year,display_name_alternatives,block_key,works_api_url,updated_date,created_date,counts_clean,latest_year,pubs_in_latest_yr
2307,https://openalex.org/A5022966906,https://orcid.org/0000-0001-8675-3561,Philippe Pasquier,"['P DU PASQUIER', 'P PASQUIER', 'P Pasquier', ...","Pasquier, Philippe",238,3062,"{'2yr_mean_citedness': 1.0952380952380953, 'h_...",{'openalex': 'https://openalex.org/A5022966906...,[{'institution': {'id': 'https://openalex.org/...,...,"[{'id': '41008148', 'wikidata': 'https://www.w...","[{'year': 1952, 'works_count': 1, 'oa_works_co...","['P DU PASQUIER', 'P PASQUIER', 'P Pasquier', ...",p pasquier,https://api.openalex.org/works?filter=author.i...,2026-05-01T12:19:04,2016-06-24T00:00:00,"[{'year': 1952, 'works_count': 1, 'oa_works_co...",2026,3


In [41]:
oa_authors_clean[oa_authors_clean['last_name'].str.contains("pasquier")]

,id,orcid,display_name,raw_author_names,full_name,works_count,cited_by_count,summary_stats,ids,affiliations,...,block_key,works_api_url,counts_clean,latest_year,pubs_in_latest_yr,name_raw,last_name,last_init,first_names,potential_matches
2261,A5022966906,0000-0001-8675-3561,Philippe Pasquier,"['P DU PASQUIER', 'P PASQUIER', 'P Pasquier', ...","Pasquier, Philippe",238,3062,"{'2yr_mean_citedness': 1.0952380952380953, 'h_...",{'openalex': 'https://openalex.org/A5022966906...,[{'institution': {'id': 'https://openalex.org/...,...,p pasquier,https://api.openalex.org/works?filter=author.i...,"[{'year': 1952, 'works_count': 1, 'oa_works_co...",2026,3,philippe pasquier,pasquier,p,['philippe'],[]


In [65]:
len(oa_authors_raw)

28716

In [66]:
len(oa_authors_clean)

20049

### first catch and merge all exact matches 
THRESHOLD = 100

In [42]:
# first catch and merge all exact matches 
THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
filepath = f"./intermediate_results/perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(oa_authors_clean.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['id'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
perfect_matches.to_csv(filepath)

#display(perfect_matches)

100%|██████████| 48/48 [07:14<00:00,  9.05s/it]


In [43]:
len(perfect_matches)

7965

In [44]:
# Build adjacency list
graph = defaultdict(set)

for _, row in perfect_matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)


In [45]:
oa_authors_clean.columns

Index(['id', 'orcid', 'display_name', 'raw_author_names', 'full_name',
       'works_count', 'cited_by_count', 'summary_stats', 'ids', 'affiliations',
       'last_known_institutions', 'topics', 'display_name_alternatives',
       'block_key', 'works_api_url', 'counts_clean', 'latest_year',
       'pubs_in_latest_yr', 'name_raw', 'last_name', 'last_init',
       'first_names', 'potential_matches'],
      dtype='object')

In [46]:
# initialize deduplicated data table
oa_authors_deduplicated = oa_authors_clean

test = oa_authors_clean

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = oa_authors_deduplicated[
        oa_authors_deduplicated['id'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['id'] = test.loc[0]['id']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'id']]
        )

        # 2. resolve oricd
        if pd.notna(test.loc[0]['orcid']):
            new_row['orcid'] = test.loc[0]['orcid']
        elif pd.notna(test.loc[j, 'orcid']):
            new_row['orcid'] = test.loc[j]['orcid']
        else: 
            new_row['orcid'] = np.nan

        # 3. leave display name as is

        # 4. merge alt names 
        new_row['display_name_alternatives'] = new_row['display_name_alternatives'] + test.loc[j]['display_name_alternatives']
        
        # 5. sum works count
        new_row['works_count'] = new_row['works_count'] + test.loc[j]['works_count']
        
        # 6. sum citations count 
        new_row['cited_by_count'] = new_row['cited_by_count'] + test.loc[j]['cited_by_count']
        
        # 7. deal with summary stats (for later, for now keep biggest only)
        
        # 8. deal with ids (also for later)
        
        # 9. affiliations
        new_row['affiliations'] = new_row['affiliations'] + test.loc[j]['affiliations']
        
        # 10. last known inst
        #new_row['last_known_institutions'] = new_row['last_known_institutions'] + test.loc[j]['last_known_institutions']
        
        # 11. topics
        new_row['topics'] = new_row['topics'] + test.loc[j]['topics']
        
        # 12. works api
        new_row['works_api_url'] = new_row['works_api_url'] + test.loc[j]['works_api_url']
        
        # 13. counts
        new_row.at[idx, 'counts_clean'] = merge_year_dict_lists(
            new_row.at[idx, 'counts_clean'],
            test.loc[j, 'counts_clean']
        )

        # 14. latest year 
        new_row['latest_year'] = max(new_row['latest_year'][0], test['latest_year'][j])

        # 15. works in latest year
        latest_count = new_row.at[idx, 'counts_clean'][-1]['oa_works_count']
        new_row.at[idx, 'pubs_in_latest_yr'] = latest_count

        # 16. name is same 

        # 17. last name is same 

        # 18. last init is same 

        # 19. first names are same 

        # 20. potential matches dealt with above 

    oa_authors_deduplicated = oa_authors_deduplicated[~oa_authors_deduplicated["id"].isin(ids_test)]
    oa_authors_deduplicated = pd.concat([oa_authors_deduplicated, new_row], ignore_index=True)

100%|██████████| 1001/1001 [00:30<00:00, 33.08it/s]


In [47]:
#oa_authors_deduplicated

#### now merge imperfect matches

test different values of the threshold to see what looks best

In [48]:
# Set the fuzzy-match threshold here
THRESHOLD = 85

imperfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filepath = f"./intermediate_results/oa_imperfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(oa_authors_deduplicated.groupby('last_init')):
    names = group['name_raw'].to_list()
    ids = group['id'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if THRESHOLD <= similarity_score < 100:
                imperfect_matches.append({
                    'name1': name1,
                    'name2': name2,
                    'id1': ids[i],
                    'id2': ids[j],
                    'similarity_score': similarity_score
                })

matches = pd.DataFrame(imperfect_matches)
matches.to_csv(filepath)
print(f"Found {len(matches)} imperfect matches at threshold >= {THRESHOLD}")


100%|██████████| 48/48 [03:45<00:00,  4.70s/it]

Found 1996 imperfect matches at threshold >= 85


### ~~threshold used is 85~~ matched all correct matches found using the 75% threshold
See excel sheet for more detail 

In [49]:
# Build adjacency list
graph = defaultdict(set)

for _, row in matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)


In [51]:
# initialize deduplicated data table
oa_authors_deduplicated_2 = oa_authors_deduplicated

test = oa_authors_deduplicated

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = oa_authors_deduplicated_2[
        oa_authors_deduplicated_2['id'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['id'] = test.loc[0]['id']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'id']]
        )

        # 2. resolve oricd
        if pd.notna(test.loc[0]['orcid']):
            new_row['orcid'] = test.loc[0]['orcid']
        elif pd.notna(test.loc[j, 'orcid']):
            new_row['orcid'] = test.loc[j]['orcid']
        else: 
            new_row['orcid'] = np.nan

        # 3. leave display name as is

        # 4. merge alt names 
        new_row['display_name_alternatives'] = new_row['display_name_alternatives'] + test.loc[j]['display_name_alternatives']
        
        # 5. sum works count
        new_row['works_count'] = new_row['works_count'] + test.loc[j]['works_count']
        
        # 6. sum citations count 
        new_row['cited_by_count'] = new_row['cited_by_count'] + test.loc[j]['cited_by_count']
        
        # 7. deal with summary stats (for later, for now keep biggest only)
        
        # 8. deal with ids (also for later)
        
        # 9. affiliations
        new_row['affiliations'] = new_row['affiliations'] + test.loc[j]['affiliations']
        
        # 10. last known inst
        #new_row['last_known_institutions'] = new_row['last_known_institutions'] + test.loc[j]['last_known_institutions']
        
        # 11. topics
        new_row['topics'] = new_row['topics'] + test.loc[j]['topics']
        
        # 12. works api
        new_row['works_api_url'] = new_row['works_api_url'] + test.loc[j]['works_api_url']
        
        # 13. counts
        new_row.at[idx, 'counts_clean'] = merge_year_dict_lists(
            new_row.at[idx, 'counts_clean'],
            test.loc[j, 'counts_clean']
        )

        # 14. latest year 
        new_row['latest_year'] = max(new_row['latest_year'][0], test['latest_year'][j])

        # 15. works in latest year
        latest_count = new_row.at[idx, 'counts_clean'][-1]['oa_works_count']
        new_row.at[idx, 'pubs_in_latest_yr'] = latest_count

        # 16. keep top name

        # 17. keep top last name

        # 18. keep top last init 

        # 19. keep top first names 

        # 20. potential matches dealt with above 

    oa_authors_deduplicated_2 = oa_authors_deduplicated_2[~oa_authors_deduplicated_2["id"].isin(ids_test)]
    oa_authors_deduplicated_2 = pd.concat([oa_authors_deduplicated_2, new_row], ignore_index=True)

100%|██████████| 947/947 [00:28<00:00, 33.62it/s]


In [52]:
print(len(oa_authors_deduplicated_2))

16351


In [53]:
#oa_authors_deduplicated_2

oa_authors_deduplicated_2.to_csv("oa_authors_20-24_deduplicated.csv", index=False)

### 2. Read and clean SciVal author data for SFU

Again, limit only to authors who have published at least once since 2020.

In [54]:
# read SciVal data
sv_authors_raw = pd.read_excel("SciVal SFU 14-24 full author list.xlsx", engine="openpyxl", skiprows=11)

print(len(sv_authors_raw))
#sv_authors_raw

10637


In [55]:
print(len(sv_authors_raw.columns))
sv_authors_raw.columns

12


Index(['Name', 'Scholarly Output', 'Most recent publication', 'Citations',
       'Citations per Publication', 'Field-Weighted Citation Impact',
       'h-index', 'Output in Top 10% Citation Percentiles (field-weighted)',
       'Oldest publication (since 1996)', 'Scopus author ID',
       'Scopus author profile', 'Primary author affiliation*'],
      dtype='object')

In [56]:
# clean scival data

sv_authors_clean = sv_authors_raw

# drop unnecessary cols
sv_authors_clean = sv_authors_clean.drop(columns=['Output in Top 10% Citation Percentiles (field-weighted)', 'Oldest publication (since 1996)'])

# keep only authors who published since 2020
sv_authors_old = sv_authors_clean[sv_authors_clean['Most recent publication'] < 2020].reset_index().drop(columns=['index'])
sv_authors_clean = sv_authors_clean[sv_authors_clean['Most recent publication'] >= 2020].reset_index().drop(columns=['index'])

# clean scholarly output
sv_authors_clean['Scholarly Output'] = sv_authors_clean['Scholarly Output'].apply(lambda x: int(x))

# clean most recent pubs 
sv_authors_clean['Most recent publication'] = sv_authors_clean['Most recent publication'].apply(lambda x: int(x))

# clean citations 
sv_authors_clean['Citations'] = sv_authors_clean['Citations'].apply(lambda x: int(x))

# clean h-index 
sv_authors_clean['h-index'] = sv_authors_clean['h-index'].apply(lambda x: int(x))

# clean h-index 
sv_authors_clean['h-index'] = sv_authors_clean['h-index'].apply(lambda x: int(x))

# clean id
sv_authors_clean['Scopus author ID'] = sv_authors_clean['Scopus author ID'].apply(lambda x: int(x))

# deal with names
sv_authors_clean['last_name'] = sv_authors_clean['Name'].apply(lambda x: str(x).split(',')[0].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['last_init'] = sv_authors_clean['last_name'].apply(lambda x: str(x)[0].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['first_names'] = sv_authors_clean['Name'].apply(lambda x: str(x).split(',')[-1].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['name_raw'] = sv_authors_clean['first_names'] + ' ' + sv_authors_clean['last_name']
sv_authors_clean['potential_matches'] = [[] for _ in range(len(sv_authors_clean))]

#sv_authors_clean

### first catch and merge all exact matches
THRESHOLD = 100 

In [57]:
# first catch and merge all exact matches 
THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
filepath = f"./intermediate_results/sv_perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(sv_authors_clean.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['Scopus author ID'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
perfect_matches.to_csv(filepath)

#display(perfect_matches)


100%|██████████| 29/29 [00:34<00:00,  1.20s/it]


In [58]:
# Build adjacency list
graph = defaultdict(set)

for _, row in perfect_matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)

In [59]:
# initialize deduplicated data table
sv_authors_deduplicated = sv_authors_clean

test = sv_authors_clean

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = sv_authors_deduplicated[
        sv_authors_deduplicated['Scopus author ID'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['Scopus author ID'] = test.loc[0]['Scopus author ID']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'Scopus author ID']]
        )
        
        # 5. sum works count
        new_row['Scholarly Output'] = new_row['Scholarly Output'] + test.loc[j]['Scholarly Output']
        
        # 6. sum citations count 
        new_row['Citations'] = new_row['Citations'] + test.loc[j]['Citations']
        
        # 7. citations per publication
        new_row['Citations per Publication'] = new_row['Citations']/new_row['Scholarly Output']
        
        # 8. deal with fwci later

        # 9. deal with h-index later
        
        # 10. leave author profile for now
        
        # 10. last known inst
        #new_row['Primary author affiliation*'] = [[new_row['Primary author affiliation*']], [test.loc[j]['Primary author affiliation*']]]
        
        # 16. name is same 

        # 17. last name is same 

        # 18. last init is same 

        # 19. first names are same 

        # 20. potential matches dealt with above 

    sv_authors_deduplicated = sv_authors_deduplicated[~sv_authors_deduplicated["Scopus author ID"].isin(ids_test)]
    sv_authors_deduplicated = pd.concat([sv_authors_deduplicated, new_row], ignore_index=True)

100%|██████████| 165/165 [00:00<00:00, 175.14it/s]


#### now merge imperfect matches

test different values of the threshold to see what looks best

### ~~threshold used is 85~~ see above
See excel sheet for more detail 

In [60]:
matches = pd.read_excel("sv name resolution threshold check.xlsx", sheet_name="75").drop(columns=['Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9'])

matches = matches[matches['ok/no? '] == "ok"]

print(len(matches))

#display(matches)

131


In [61]:
# Build adjacency list
graph = defaultdict(set)

for _, row in matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)


In [62]:
# initialize deduplicated data table
sv_authors_deduplicated_2 = sv_authors_deduplicated

test = sv_authors_deduplicated

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = sv_authors_deduplicated_2[
        sv_authors_deduplicated_2['Scopus author ID'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['Scopus author ID'] = test.loc[0]['Scopus author ID']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'Scopus author ID']]
        )
        
        # 5. sum works count
        new_row['Scholarly Output'] = new_row['Scholarly Output'] + test.loc[j]['Scholarly Output']
        
        # 6. sum citations count 
        new_row['Citations'] = new_row['Citations'] + test.loc[j]['Citations']
        
        # 7. citations per publication
        new_row['Citations per Publication'] = new_row['Citations']/new_row['Scholarly Output']
        
        # 8. deal with fwci later

        # 9. deal with h-index later
        
        # 10. leave author profile for now
        
        # 10. last known inst
        #new_row['Primary author affiliation*'] = [[new_row['Primary author affiliation*']], [test.loc[j]['Primary author affiliation*']]]
        
        # 16. name is same 

        # 17. last name is same 

        # 18. last init is same 

        # 19. first names are same 

        # 20. potential matches dealt with above 

    sv_authors_deduplicated_2 = sv_authors_deduplicated_2[~sv_authors_deduplicated_2["Scopus author ID"].isin(ids_test)]
    sv_authors_deduplicated_2 = pd.concat([sv_authors_deduplicated_2, new_row], ignore_index=True)

100%|██████████| 115/115 [00:00<00:00, 149.46it/s]


In [63]:
print(len(sv_authors_deduplicated_2))

6362


In [64]:
#sv_authors_deduplicated_2

sv_authors_deduplicated_2.to_csv("sv_authors_20-24_deduplicated.csv", index=False)